<a href="https://colab.research.google.com/github/MattManson/lastfm-music-pipeline/blob/main/lastfm_medallion_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages io.delta:delta-spark_2.12:3.3.0 pyspark-shell'
print("[SETUP] Delta environment configured")

[SETUP] Delta environment configured


In [45]:
!pip install delta-spark==3.3.0

In [46]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("lastfm-pipeline") \
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .master("local[*]") \
    .getOrCreate()

print(spark.version)

3.5.8


In [47]:
import requests
import json
import time
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from google.colab import userdata

# API config
API_KEY = userdata.get('LASTFM_API_KEY')
BASE_URL = "http://ws.audioscrobbler.com/2.0/"

print("[SETUP] Imports and API config complete")

[SETUP] Imports and API config complete


In [48]:
BASE_PATH = "/tmp/lastfm_pipeline"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"

print(f"Bronze: {BRONZE_PATH}")
print(f"Silver: {SILVER_PATH}")
print(f"Gold: {GOLD_PATH}")

Bronze: /tmp/lastfm_pipeline/bronze
Silver: /tmp/lastfm_pipeline/silver
Gold: /tmp/lastfm_pipeline/gold


In [49]:
def get_top_tracks(country, limit=50):
    """
    Fetch top tracks for a given country from Last.fm.
    country: country name e.g. 'united kingdom'
    limit: number of tracks to return, max 50 per page
    """
    url = f"{BASE_URL}?method=geo.gettoptracks&country={country}&limit={limit}&api_key={API_KEY}&format=json"
    response = requests.get(url)
    data = response.json()
    print(f"[INGEST] Fetched {len(data['tracks']['track'])} tracks for {country}")
    return data

def get_artist_tags(artist_name, snapshot_date):
    """
    Fetch genre tags for a single artist from Last.fm.
    Returns list of (artist_name, tag_name, tag_count) tuples.
    """
    url = f"{BASE_URL}?method=artist.gettoptags&artist={artist_name}&api_key={API_KEY}&format=json"
    response = requests.get(url)
    data = response.json()

    if "toptags" not in data:
        print(f"[INGEST] No tags found for {artist_name}")
        return []

    tags = data["toptags"]["tag"]
    parsed_tags = []
    for tag in tags:
        parsed_tags.append((
            artist_name,
            tag["name"],
            int(tag["count"]),
            snapshot_date
        ))
    return parsed_tags

def get_all_artist_tags(parsed_tracks, snapshot_date):
    """
    Fetch tags for every unique artist in our tracks list.
    Includes rate limiting to respect Last.fm's API limits.
    """
    artists = list(set([track[1] for track in parsed_tracks]))
    all_tags = []
    for artist in artists:
        print(f"[INGEST] Fetching tags for {artist}...")
        tags = get_artist_tags(artist, snapshot_date)
        all_tags.extend(tags)
        time.sleep(0.2)
    print(f"[INGEST] Fetched {len(all_tags)} tag records for {len(artists)} artists")
    return all_tags

print("[SETUP] Ingestion functions defined")

[SETUP] Ingestion functions defined


In [50]:
def write_bronze(raw_data, source):
    """
    Write raw API response to the bronze Delta table.
    Stores data exactly as received with ingestion timestamp.
    Appends each run to build up history over time.
    """
    ingested_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    raw_json_string = json.dumps(raw_data)

    bronze_data = [(source, ingested_at, raw_json_string)]
    bronze_columns = ["source", "ingested_at", "raw_json"]

    bronze_df = spark.createDataFrame(bronze_data, bronze_columns)

    bronze_df.write \
        .format("delta") \
        .mode("append") \
        .save(f"{BRONZE_PATH}/raw_api_responses")

    print(f"[BRONZE] Written {source} at {ingested_at}")

print("[SETUP] Bronze functions defined")

[SETUP] Bronze functions defined


In [74]:
track_schema = StructType([
    StructField("track_name", StringType(), True),
    StructField("artist_name", StringType(), True),
    StructField("duration_seconds", IntegerType(), True),
    StructField("listeners", IntegerType(), True),
    StructField("chart_rank", IntegerType(), True),
    StructField("snapshot_date", StringType(), True),
    StructField("country", StringType(), True)
])

artist_tag_schema = StructType([
    StructField("artist_name", StringType(), True),
    StructField("tag_name", StringType(), True),
    StructField("tag_count", IntegerType(), True),
    StructField("snapshot_date", StringType(), True)
])

def parse_tracks(raw_data, snapshot_date, country):
    """
    Extract track records from raw Last.fm API response.
    Flattens nested JSON into typed tuples ready for DataFrame creation.
    """
    tracks = raw_data["tracks"]["track"]
    parsed_tracks = []
    for track in tracks:
        duration = int(track["duration"]) if track["duration"] else 0
        parsed_tracks.append((
            track["name"],
            track["artist"]["name"],
            duration,
            int(track["listeners"]),
            int(track["@attr"]["rank"]) + 1,
            snapshot_date,
            country
        ))
    print(f"[SILVER] Parsed {len(parsed_tracks)} tracks")
    return parsed_tracks

def write_silver_tracks(parsed_tracks, country):
    """
    Write parsed track data to the silver Delta table.
    Idempotent - skips write if data for this snapshot date already exists.
    """
    from delta.tables import DeltaTable
    import os

    snapshot_date = parsed_tracks[0][5]
    silver_path = f"{SILVER_PATH}/tracks"

    if os.path.exists(silver_path):
        existing = spark.read.format("delta").load(silver_path)
        already_loaded = existing.filter(
            (existing.snapshot_date == snapshot_date) &
            (existing.country == country)
        ).count()

        if already_loaded>0:
            print(f"[SILVER] Tracks for {snapshot_date} already exists - skipping write")
            return

    silver_df = spark.createDataFrame(parsed_tracks, track_schema)
    silver_df.write \
        .format("delta") \
        .mode("append") \
        .save(f"{SILVER_PATH}/tracks")
    print(f"[SILVER] Written {len(parsed_tracks)} tracks to silver")

def write_silver_tags(all_tags, snapshot_date):
    """
    Write artist tag data to the silver Delta table.
    Idempotent - skips write if data for this snapshot already exists
    """
    import os

    silver_path = f"{SILVER_PATH}/artist_tags"

    if os.path.exists(silver_path):
        existing = spark.read.format("delta").load(silver_path)
        already_loaded = existing.filter(
            existing.snapshot_date == snapshot_date
        ).count()

        if already_loaded >0:
            print(f"[SILVER] tags for {snapshot_date} already exist - skipping write")
            return

    tags_df = spark.createDataFrame(all_tags, artist_tag_schema)
    tags_df.write \
        .format("delta") \
        .mode("append") \
        .save(f"{SILVER_PATH}/artist_tags")
    print(f"[SILVER] Written {len(all_tags)} tag records to silver")

print("[SETUP] Silver functions defined")

[SETUP] Silver functions defined


In [65]:
def build_gold_genre_summary(snapshot_date):
    """
    Join silver tracks and artist tags to produce a genre summary.
    Gold layer - aggregated, business ready data for analysis.
    Idempotent - skips write if data for this snapshot date already exists.
    """
    from pyspark.sql.functions import sum, count, avg, col, lit
    import os

    gold_path = f"{GOLD_PATH}/genre_summary"

    if os.path.exists(gold_path):
        existing = spark.read.format("delta").load(gold_path)
        already_loaded = existing.filter(
            existing.snapshot_date == snapshot_date
        ).count()
        if already_loaded > 0:
            print(f"[GOLD] Genre summary for {snapshot_date} already exists - skipping write")
            return existing.filter(existing.snapshot_date == snapshot_date)

    tracks = spark.read.format("delta").load(f"{SILVER_PATH}/tracks")
    tags = spark.read.format("delta").load(f"{SILVER_PATH}/artist_tags")

    tracks = tracks.filter(tracks.snapshot_date == snapshot_date)
    tags = tags.filter(tags.tag_count > 10)

    joined = tracks.join(tags, on="artist_name", how="left")

    gold_df = joined.groupBy("tag_name") \
        .agg(
            count("track_name").alias("track_count"),
            sum("listeners").alias("total_listeners"),
            avg("chart_rank").alias("avg_chart_rank")
        ) \
        .orderBy(col("total_listeners").desc()) \
        .withColumn("snapshot_date", lit(snapshot_date))

    gold_df.write \
        .format("delta") \
        .mode("append") \
        .save(gold_path)

    print(f"[GOLD] Genre summary written for {snapshot_date}")
    return gold_df

print("[SETUP] Gold functions defined")

[SETUP] Gold functions defined


In [75]:
def run_pipeline(countries=["united kingdom"], snapshot_date=None):
    """
    Runs the full Last.FM medallion pipeline end to end.
    1 - Ingest top tracks per country to bronze
    2 - Ingest artist tags to bronze
    3 - Parse tracks to silver
    4 - Parse tags to silver
    5 - Build gold genre summary

    countries:      list of countries to ingest
    snapshot_date:  date string YYYY-MM-DD, defaults to today
    """

    if snapshot_date is None:
        snapshot_date = datetime.now().strftime("%Y-%m-%d")

    print(f"[PIPELINE] Starting run for {snapshot_date}")
    print(f"[PIPELINE] Countries: {countries}")

    all_parsed_tracks = []
    all_parsed_tags = []

    for country in countries:
        #Bronze - raw tracks
        raw_tracks = get_top_tracks(country, limit=50)
        write_bronze(raw_tracks, f"lastfm_geo_toptracks_{country.replace(' ','_')}")

        #Silver - parsed tracks
        parsed_tracks = parse_tracks(raw_tracks, snapshot_date, country)
        all_parsed_tracks.extend(parsed_tracks)

    all_parsed_tags = get_all_artist_tags(all_parsed_tracks, snapshot_date)

    #Bronze - raw tags per artist
    for artist_name in list(set([track[1] for track in all_parsed_tracks])):
        raw_tags = get_artist_tags(artist_name, snapshot_date)
        if raw_tags:
            write_bronze(
                {"artist": artist_name, "tags": raw_tags},
                f"lastfm_artist_tags_{artist_name.replace(' ','_')}"
            )

    #Silver - write tracks and tags
    write_silver_tracks(all_parsed_tracks, country)
    write_silver_tags(all_parsed_tags, snapshot_date)

    #Gold
    gold_df = build_gold_genre_summary(snapshot_date)
    return gold_df

print("[SETUP] Pipeline function defined")

[SETUP] Pipeline function defined


In [78]:
gold_df = run_pipeline(
    countries=["united kingdom", "united states", "germany"],
    snapshot_date=None
)

gold_df.show(truncate=50)

[PIPELINE] Starting run for 2026-03-25
[PIPELINE] Countries: ['united kingdom', 'united states', 'germany']
[INGEST] Fetched 50 tracks for united kingdom
[BRONZE] Written lastfm_geo_toptracks_united_kingdom at 2026-03-25 17:29:37
[SILVER] Parsed 50 tracks
[INGEST] Fetched 50 tracks for united states
[BRONZE] Written lastfm_geo_toptracks_united_states at 2026-03-25 17:29:38
[SILVER] Parsed 50 tracks
[INGEST] Fetched 50 tracks for germany
[BRONZE] Written lastfm_geo_toptracks_germany at 2026-03-25 17:29:38
[SILVER] Parsed 50 tracks
[INGEST] Fetching tags for Frank Ocean...
[INGEST] Fetching tags for Lady Gaga...
[INGEST] Fetching tags for Sam Fender...
[INGEST] Fetching tags for Weezer...
[INGEST] Fetching tags for Dominic Fike...
[INGEST] Fetching tags for Julia Wolf...
[INGEST] Fetching tags for The Marías...
[INGEST] Fetching tags for The Neighbourhood...
[INGEST] Fetching tags for Sean Paul...
[INGEST] Fetching tags for bôa...
[INGEST] Fetching tags for The Goo Goo Dolls...
[INGEST] 